<a href="https://colab.research.google.com/github/jadeacevedo/LSTM-Stock-Forecaster-Multivariate/blob/version.2/v2_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
"""
=================================================================
  STOCK PRICE PREDICTION USING RNN AND LSTM
  Apple Inc. (AAPL) — NumPy Vectorized Implementation
=================================================================
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings, shutil
warnings.filterwarnings('ignore')
np.random.seed(42)

print("=" * 62)
print("  STOCK PRICE PREDICTION — LSTM & RNN (NumPy from scratch)")
print("=" * 62)

# ══════════════════════════════════════════════════════════════
# 1. GENERATE SYNTHETIC AAPL DATA
# ══════════════════════════════════════════════════════════════
print("\n[1/6] Generating AAPL-like stock data (Geometric Brownian Motion)…")

N = 800          # trading days
dt = 1/252; mu = 0.12; sigma = 0.22
prices = [150.0]
for _ in range(N-1):
    prices.append(prices[-1]*np.exp((mu-.5*sigma**2)*dt + sigma*np.sqrt(dt)*np.random.randn()))
prices = np.array(prices) * (1 + .03*np.sin(np.linspace(0,4*np.pi,N)))

dates = pd.bdate_range('2020-01-01', periods=N)
dv    = sigma * prices * np.sqrt(dt)
df = pd.DataFrame({
    'Open':   prices + np.random.normal(0,dv*.3),
    'High':   prices + np.abs(np.random.normal(0,dv*.7)),
    'Low':    prices - np.abs(np.random.normal(0,dv*.7)),
    'Close':  prices,
    'Volume': np.random.randint(50_000_000,120_000_000,N),
}, index=dates)
df['High'] = df[['Open','High','Close']].max(1)
df['Low']  = df[['Open','Low', 'Close']].min(1)

# Technical indicators
def ema(s, span): return s.ewm(span=span,adjust=False).mean()
def rsi(s, p=14):
    d=s.diff(); g=d.clip(lower=0).rolling(p).mean(); l=(-d.clip(upper=0)).rolling(p).mean()
    return 100-100/(1+g/(l+1e-9))

df['EMA_10']   = ema(df['Close'],10)
df['EMA_50']   = ema(df['Close'],50)
df['RSI_14']   = rsi(df['Close'])
df['MA_20']    = df['Close'].rolling(20).mean()
df['BB_U']     = df['MA_20'] + 2*df['Close'].rolling(20).std()
df['BB_L']     = df['MA_20'] - 2*df['Close'].rolling(20).std()
df['Return']   = df['Close'].pct_change()
df['Volatility']= df['Return'].rolling(10).std()
df.dropna(inplace=True)

print(f"   {len(df)} trading days  |  {df.index[0].date()} → {df.index[-1].date()}")
print(df[['Open','High','Low','Close','Volume']].head().to_string())

# ══════════════════════════════════════════════════════════════
# 2. PREPROCESSING
# ══════════════════════════════════════════════════════════════
print("\n[2/6] Preprocessing & building sliding-window sequences…")

SEQ = 30                        # look-back window
SPLIT = 0.80

def scale(x):
    mn,mx = x.min(0), x.max(0)
    return (x-mn)/(mx-mn+1e-9), mn, mx

def inv(xs, mn, mx):  return xs*(mx-mn)+mn

def make_seqs(data, seq):
    X = np.array([data[i-seq:i] for i in range(seq,len(data))])
    y = data[seq:, 0]
    return X, y

# Univariate
cv = df['Close'].values.reshape(-1,1)
cs,c0,c1 = scale(cv)
n_tr = int(len(cs)*SPLIT)
Xtr_u,ytr_u = make_seqs(cs[:n_tr], SEQ)
Xte_u,yte_u = make_seqs(np.vstack([cs[n_tr-SEQ:],cs[n_tr:]]), SEQ)
# shape → (samples, SEQ, 1)

# Multivariate
feats = ['Close','EMA_10','EMA_50','RSI_14','Volatility']
mv = df[feats].values
ms,m0,m1 = scale(mv)
Xtr_m,ytr_m = make_seqs(ms[:n_tr], SEQ)
Xte_m,yte_m = make_seqs(np.vstack([ms[n_tr-SEQ:],ms[n_tr:]]), SEQ)

print(f"   Seq len={SEQ}  |  Train={len(Xtr_u)}  Test={len(Xte_u)}")

# ══════════════════════════════════════════════════════════════
# 3. VECTORIZED RNN (processes full batch at once)
# ══════════════════════════════════════════════════════════════
print("\n[3/6] Training Simple RNN (batch-vectorized)…")

class VanillaRNN:
    """
    Vectorized SimpleRNN trained with truncated BPTT.
    Input: (batch, T, F)  →  Output: (batch,)
    """
    def __init__(self, F, H=24, lr=5e-3):
        self.H=H; self.lr=lr
        s=0.08
        self.Wxh=np.random.randn(F,H)*s
        self.Whh=np.random.randn(H,H)*s
        self.bh =np.zeros(H)
        self.Wy =np.random.randn(H,1)*s
        self.by =np.zeros(1)
        self._adam_init()

    def _adam_init(self):
        self.ms={k:np.zeros_like(v) for k,v in self._pdict().items()}
        self.vs={k:np.zeros_like(v) for k,v in self._pdict().items()}
        self.step=0

    def _pdict(self): return {'Wxh':self.Wxh,'Whh':self.Whh,'bh':self.bh,'Wy':self.Wy,'by':self.by}

    def _forward(self, X):
        B,T,F = X.shape
        h = np.zeros((B,self.H))
        hs=[]
        for t in range(T):
            h = np.tanh(X[:,t,:]@self.Wxh + h@self.Whh + self.bh)
            hs.append(h)
        return hs, h@self.Wy + self.by   # (B,1)

    def _loss_grad(self, X, y):
        hs, yp = self._forward(X)
        yp = yp.flatten()
        err = yp - y                      # (B,)
        loss= np.mean(err**2)
        dh  = (err[:,None]/len(y)) @ self.Wy.T   # (B,H)
        dWy = hs[-1].T @ (err[:,None]/len(y))
        dby = err.mean(keepdims=True)
        dWxh=np.zeros_like(self.Wxh)
        dWhh=np.zeros_like(self.Whh)
        dbh =np.zeros(self.H)
        T=len(hs)
        for t in reversed(range(T)):
            dt = dh * (1-hs[t]**2)
            dbh  += dt.mean(0)
            dWxh += X[:,t,:].T @ dt / len(y)
            dWhh += (hs[t-1] if t>0 else np.zeros_like(hs[0])).T @ dt / len(y)
            dh    = dt @ self.Whh.T
        grads={'Wxh':np.clip(dWxh,-1,1),'Whh':np.clip(dWhh,-1,1),
               'bh':np.clip(dbh,-1,1),'Wy':np.clip(dWy,-1,1),'by':np.clip(dby,-1,1)}
        return loss, grads, yp

    def _adam(self, grads):
        self.step+=1
        b1,b2,eps=0.9,0.999,1e-8
        params=self._pdict()
        for k,g in grads.items():
            self.ms[k]=b1*self.ms[k]+(1-b1)*g
            self.vs[k]=b2*self.vs[k]+(1-b2)*g**2
            params[k]-=self.lr*(self.ms[k]/(1-b1**self.step))/(np.sqrt(self.vs[k]/(1-b2**self.step))+eps)

    def fit(self, X, y, Xv, yv, epochs=30, batch=64):
        hist={'tr':[],'val':[]}
        for ep in range(epochs):
            idx=np.random.permutation(len(X))
            ep_loss=0; nb=0
            for s in range(0,len(X),batch):
                bx=X[idx[s:s+batch]]; by_=y[idx[s:s+batch]]
                loss,grads,_=self._loss_grad(bx,by_)
                self._adam(grads); ep_loss+=loss; nb+=1
            _,_,vp=self._loss_grad(Xv,yv)
            vl=np.mean((vp-yv)**2)
            hist['tr'].append(ep_loss/nb); hist['val'].append(vl)
            if (ep+1)%10==0:
                print(f"   RNN  ep {ep+1:3d}/{epochs}  train={ep_loss/nb:.5f}  val={vl:.5f}")
        return hist

    def predict(self, X):
        _,yp=self._forward(X); return yp.flatten()


# ══════════════════════════════════════════════════════════════
# 4. VECTORIZED LSTM
# ══════════════════════════════════════════════════════════════
print()

class VectorizedLSTM:
    """
    Batch-vectorized LSTM.
    Weights: concatenated [h,x] → [f,i,g,o] in one matrix multiply.
    """
    def __init__(self, F, H=32, lr=3e-3):
        self.H=H; self.lr=lr; F_=F+H
        s=0.05
        self.W =np.random.randn(4*H, F_)*s   # gate weights
        self.b =np.zeros(4*H)
        self.Wy=np.random.randn(H,1)*s
        self.by=np.zeros(1)
        self._adam_init()

    def _adam_init(self):
        self.ms={k:np.zeros_like(v) for k,v in self._pdict().items()}
        self.vs={k:np.zeros_like(v) for k,v in self._pdict().items()}
        self.step=0

    def _pdict(self): return {'W':self.W,'b':self.b,'Wy':self.Wy,'by':self.by}

    @staticmethod
    def _sig(x): x=np.clip(x,-10,10); return 1/(1+np.exp(-x))

    def _forward(self, X):
        B,T,F=X.shape; H=self.H
        h=np.zeros((B,H)); c=np.zeros((B,H))
        caches=[]
        for t in range(T):
            z   = np.hstack([h,X[:,t,:]])@self.W.T + self.b   # (B,4H)
            f   = self._sig(z[:,:H])
            i   = self._sig(z[:,H:2*H])
            g   = np.tanh(z[:,2*H:3*H])
            o   = self._sig(z[:,3*H:])
            c_n = f*c + i*g
            h_n = o*np.tanh(c_n)
            caches.append((X[:,t,:],h,c,f,i,g,o,c_n,h_n))
            h,c = h_n,c_n
        yp = h@self.Wy + self.by   # (B,1)
        return caches, h, yp

    def _loss_grad(self, X, y):
        caches, h_last, yp = self._forward(X)
        yp=yp.flatten(); B=len(y)
        err=yp-y; loss=np.mean(err**2)
        dWy = h_last.T@(err[:,None]/B)
        dby = err.mean(keepdims=True)
        dh  = (err[:,None]/B)@self.Wy.T
        dc  = np.zeros_like(dh)
        dW  = np.zeros_like(self.W)
        db  = np.zeros(4*self.H)
        H=self.H
        for x_t,h_p,c_p,f,i,g,o,c_n,h_n in reversed(caches):
            tc=np.tanh(c_n)
            do=dh*tc; dc_t=dh*o*(1-tc**2)+dc
            df=dc_t*c_p; di=dc_t*g; dg=dc_t*i; dc=dc_t*f
            dz=np.hstack([df*f*(1-f), di*i*(1-i), dg*(1-g**2), do*o*(1-o)])  # (B,4H)
            concat=np.hstack([h_p,x_t])
            dW+=dz.T@concat/B; db+=dz.mean(0)
            dh=(dz@self.W)[:,:H]
        grads={'W':np.clip(dW,-1,1),'b':np.clip(db,-1,1),
               'Wy':np.clip(dWy,-1,1),'by':np.clip(dby,-1,1)}
        return loss, grads, yp

    def _adam(self, grads):
        self.step+=1; b1,b2,eps=0.9,0.999,1e-8
        params=self._pdict()
        for k,g in grads.items():
            self.ms[k]=b1*self.ms[k]+(1-b1)*g
            self.vs[k]=b2*self.vs[k]+(1-b2)*g**2
            params[k]-=self.lr*(self.ms[k]/(1-b1**self.step))/(np.sqrt(self.vs[k]/(1-b2**self.step))+eps)

    def fit(self, X, y, Xv, yv, epochs=30, batch=64, name='LSTM'):
        hist={'tr':[],'val':[]}
        for ep in range(epochs):
            idx=np.random.permutation(len(X)); ep_loss=0; nb=0
            for s in range(0,len(X),batch):
                bx=X[idx[s:s+batch]]; by_=y[idx[s:s+batch]]
                loss,grads,_=self._loss_grad(bx,by_)
                self._adam(grads); ep_loss+=loss; nb+=1
            _,_,vp=self._loss_grad(Xv,yv); vl=np.mean((vp-yv)**2)
            hist['tr'].append(ep_loss/nb); hist['val'].append(vl)
            if (ep+1)%10==0:
                print(f"   {name} ep {ep+1:3d}/{epochs}  train={ep_loss/nb:.5f}  val={vl:.5f}")
        return hist

    def predict(self, X):
        _,_,yp=self._forward(X); return yp.flatten()


# ─── Train all models ────────────────────────
print("[4/6] Training models…\n")
EPOCHS = 40

print("   >>> Simple RNN (univariate)")
rnn = VanillaRNN(F=1, H=24, lr=5e-3)
rnn_hist = rnn.fit(Xtr_u, ytr_u, Xte_u, yte_u, epochs=EPOCHS)

print("\n   >>> LSTM Univariate")
lstm_u = VectorizedLSTM(F=1, H=32, lr=3e-3)
lstm_u_hist = lstm_u.fit(Xtr_u, ytr_u, Xte_u, yte_u, epochs=EPOCHS, name='LSTM-Uni')

print("\n   >>> LSTM Multivariate (Close + EMA + RSI + Volatility)")
lstm_m = VectorizedLSTM(F=5, H=40, lr=2e-3)
lstm_m_hist = lstm_m.fit(Xtr_m, ytr_m, Xte_m, yte_m, epochs=EPOCHS, name='LSTM-Multi')

# ── Inverse-scale predictions ──────────────
def isc(arr): return inv(arr.reshape(-1,1), c0, c1).flatten()
def isc_m(arr): return arr*(m1[0]-m0[0])+m0[0]

rnn_true  = isc(yte_u)
rnn_pred  = isc(rnn.predict(Xte_u))
lu_pred   = isc(lstm_u.predict(Xte_u))
lm_true   = isc_m(yte_m)
lm_pred   = isc_m(lstm_m.predict(Xte_m))

# ══════════════════════════════════════════════════════════════
# 5. EVALUATION
# ══════════════════════════════════════════════════════════════
print("\n[5/6] Evaluation metrics…")

def metrics(name, yt, yp):
    res = yt-yp; ss_r=np.sum(res**2); ss_t=np.sum((yt-yt.mean())**2)
    return {'Model':name,
            'RMSE': np.sqrt(np.mean(res**2)),
            'MAE':  np.mean(np.abs(res)),
            'MAPE': np.mean(np.abs(res/(yt+1e-9)))*100,
            'R2':   1-ss_r/(ss_t+1e-9)}

rows=[
    metrics('Simple RNN',         rnn_true, rnn_pred),
    metrics('LSTM (Univariate)',   rnn_true, lu_pred),
    metrics('LSTM (Multivariate)', lm_true,  lm_pred),
]
rdf = pd.DataFrame(rows).set_index('Model').round(4)
print("\n" + "─"*52)
print("  MODEL PERFORMANCE — TEST SET")
print("─"*52)
print(rdf.to_string())
print("─"*52)

# ══════════════════════════════════════════════════════════════
# 6. VISUALISATIONS
# ══════════════════════════════════════════════════════════════
print("\n[6/6] Generating plots…")

C={'act':'#2c3e50','rnn':'#e74c3c','lu':'#3498db','lm':'#2ecc71',
   'e10':'#e67e22','e50':'#9b59b6','rsi':'#1abc9c','bb':'#bdc3c7'}

test_idx = df.index[n_tr+SEQ-1: n_tr+SEQ-1+len(rnn_true)]

# ── P1: Data overview
fig,axes=plt.subplots(3,1,figsize=(14,10))
fig.suptitle('AAPL Stock Data Overview',fontsize=15,fontweight='bold')
ax=axes[0]
ax.plot(df.index,df['Close'], color=C['act'],lw=1.2,label='Close')
ax.plot(df.index,df['EMA_10'],color=C['e10'],lw=1, label='EMA-10',alpha=.85)
ax.plot(df.index,df['EMA_50'],color=C['e50'],lw=1, label='EMA-50',alpha=.85)
ax.fill_between(df.index,df['BB_L'],df['BB_U'],alpha=.12,color='gray',label='Bollinger Bands')
ax.set_ylabel('Price ($)'); ax.legend(fontsize=9); ax.grid(alpha=.3)
ax.set_title('Close Price + EMA & Bollinger Bands',fontsize=11)
ax=axes[1]
ax.plot(df.index,df['RSI_14'],color=C['rsi'],lw=1.2)
ax.axhline(70,color='red',  lw=1,ls='--',label='Overbought (70)')
ax.axhline(30,color='green',lw=1,ls='--',label='Oversold (30)')
ax.fill_between(df.index,df['RSI_14'],70,where=df['RSI_14']>=70,alpha=.2,color='red')
ax.fill_between(df.index,df['RSI_14'],30,where=df['RSI_14']<=30,alpha=.2,color='green')
ax.set_ylim(0,100); ax.set_ylabel('RSI-14'); ax.legend(fontsize=9); ax.grid(alpha=.3)
ax.set_title('Relative Strength Index (RSI-14)',fontsize=11)
ax=axes[2]
ax.bar(df.index,df['Volume']/1e6,color='steelblue',alpha=.7,width=1)
ax.set_ylabel('Volume (M)'); ax.set_xlabel('Date'); ax.grid(alpha=.3,axis='y')
ax.set_title('Trading Volume',fontsize=11)
plt.tight_layout(); fig.savefig('/content/prediction1.png',dpi=110,bbox_inches='tight'); plt.close()

# ── P2: Architecture diagrams
fig,axes=plt.subplots(1,2,figsize=(16,5))
fig.suptitle('Neural Network Architectures',fontsize=14,fontweight='bold')

# RNN diagram
ax=axes[0]; ax.set_xlim(0,10); ax.set_ylim(0,6); ax.axis('off')
ax.set_title('Simple RNN — Unrolled through Time',fontsize=11,fontweight='bold')
bp=dict(boxstyle='round,pad=.5',facecolor='#3498db',alpha=.85,edgecolor='white')
hp=dict(boxstyle='round,pad=.5',facecolor='#e74c3c',alpha=.85,edgecolor='white')
op=dict(boxstyle='round,pad=.5',facecolor='#2ecc71',alpha=.85,edgecolor='white')
for j,lbl in enumerate(['x(t-2)','x(t-1)','x(t)']):
    x0=1.5+j*3
    ax.text(x0,1.0,lbl,ha='center',va='center',fontsize=10,color='white',bbox=bp)
    ax.text(x0,3.2,f'h(t{j-2:+d})',ha='center',va='center',fontsize=10,color='white',bbox=hp)
    ax.annotate('',xy=(x0,2.75),xytext=(x0,1.4),arrowprops=dict(arrowstyle='->',color='#555',lw=1.5))
    if j<2:
        ax.annotate('',xy=(x0+3-.45,3.2),xytext=(x0+.45,3.2),
                    arrowprops=dict(arrowstyle='->',color='#c0392b',lw=2))
ax.text(7.5,4.7,'ŷ',ha='center',va='center',fontsize=12,color='white',bbox=op)
ax.annotate('',xy=(7.5,4.3),xytext=(7.5,3.65),arrowprops=dict(arrowstyle='->',color='#555',lw=1.5))
ax.text(1.5,5,'tanh  ⟶  Recurrent',ha='left',fontsize=9,color='gray')

# LSTM diagram
ax=axes[1]; ax.set_xlim(0,12); ax.set_ylim(0,7); ax.axis('off')
ax.set_title('LSTM Cell — Gate Architecture',fontsize=11,fontweight='bold')
gp=lambda c: dict(boxstyle='round,pad=.5',facecolor=c,alpha=.9,edgecolor='white')
gates=[('σ','Forget\nGate','#e74c3c',2.0),
       ('σ','Input\nGate', '#3498db',4.5),
       ('tanh','Cell\nUpd.',  '#9b59b6',7.0),
       ('σ','Output\nGate','#e67e22',9.5)]
for sym,lbl,col,xp in gates:
    ax.text(xp,4.0,sym,ha='center',va='center',fontsize=10,color='white',bbox=gp(col))
    ax.text(xp,3.15,lbl,ha='center',va='center',fontsize=8,color=col,style='italic')
    ax.annotate('',xy=(xp,3.55),xytext=(xp,2.4),arrowprops=dict(arrowstyle='->',color='#555',lw=1.2))
ax.annotate('',xy=(11.2,5.3),xytext=(.5,5.3),arrowprops=dict(arrowstyle='->',color='#f39c12',lw=3))
ax.text(5.8,5.8,'Cell State  C(t)',ha='center',fontsize=10,color='#f39c12',fontweight='bold')
ax.text(5.8,1.8,'x(t) + h(t-1)  [Concatenated Input]',ha='center',fontsize=10,color='#2c3e50',fontweight='bold')
ax.text(10.8,3.0,'h(t)',ha='center',fontsize=10,color='white',fontweight='bold',
        bbox=dict(boxstyle='round,pad=.4',facecolor='#2ecc71',alpha=.9,edgecolor='white'))
plt.tight_layout(); fig.savefig('/content/prediction2.png',dpi=110,bbox_inches='tight'); plt.close()

# ── P3: Loss curves
fig,axes=plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Training & Validation Loss Curves',fontsize=13,fontweight='bold')
for ax,(name,hist,color) in zip(axes,[
    ('Simple RNN',        rnn_hist,    C['rnn']),
    ('LSTM Univariate',   lstm_u_hist, C['lu']),
    ('LSTM Multivariate', lstm_m_hist, C['lm']),
]):
    ep=range(1,len(hist['tr'])+1)
    ax.plot(ep,hist['tr'],color=color, lw=2,  label='Train')
    ax.plot(ep,hist['val'],color='k', lw=1.5,ls='--',label='Val')
    ax.set_title(name,fontsize=10,fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=9); ax.grid(alpha=.3); ax.set_facecolor('#f9f9f9')
plt.tight_layout(); fig.savefig('/content/prediction3.png',dpi=110,bbox_inches='tight'); plt.close()

# ── P4: Predictions vs Actual
fig,axes=plt.subplots(3,1,figsize=(14,12))
fig.suptitle('Model Predictions vs Actual Stock Price (Test Set)',fontsize=14,fontweight='bold')
multi_test_idx = df.index[n_tr+SEQ-1: n_tr+SEQ-1+len(lm_true)]
for ax,(name,preds,yt,color,tidx) in zip(axes,[
    ('Simple RNN',         rnn_pred, rnn_true, C['rnn'], test_idx),
    ('LSTM (Univariate)',   lu_pred,  rnn_true, C['lu'],  test_idx),
    ('LSTM (Multivariate)', lm_pred,  lm_true,  C['lm'],  multi_test_idx),
]):
    n=min(len(yt),len(preds),len(tidx)); td=tidx[:n]
    ax.plot(td,yt[:n],  color=C['act'],lw=1.5,label='Actual',zorder=3)
    ax.plot(td,preds[:n],color=color,lw=1.5,ls='--',label=name,alpha=.9,zorder=4)
    r=rdf.loc[name]; ax.set_ylabel('Price ($)'); ax.legend(fontsize=9); ax.grid(alpha=.3)
    ax.set_title(f'{name}  |  RMSE=${r["RMSE"]:.2f}   MAE=${r["MAE"]:.2f}   R²={r["R2"]:.4f}',
                 fontsize=11,fontweight='bold'); ax.set_facecolor('#f9f9f9')
axes[-1].set_xlabel('Date')
plt.tight_layout(); fig.savefig('/content/prediction4.png',dpi=110,bbox_inches='tight'); plt.close()

# ── P5: Residuals
fig,axes=plt.subplots(2,3,figsize=(15,7))
fig.suptitle('Residual Analysis',fontsize=13,fontweight='bold')
for col,(name,yt,yp,color) in enumerate([
    ('Simple RNN',         rnn_true, rnn_pred, C['rnn']),
    ('LSTM Univariate',    rnn_true, lu_pred,  C['lu']),
    ('LSTM Multivariate',  lm_true,  lm_pred,  C['lm']),
]):
    n=min(len(yt),len(yp)); res=yt[:n]-yp[:n]
    axes[0,col].plot(res,color=color,lw=.9,alpha=.8)
    axes[0,col].axhline(0,color='k',lw=1,ls='--')
    axes[0,col].set_title(f'{name}\nResiduals',fontsize=10,fontweight='bold')
    axes[0,col].set_xlabel('Test Index'); axes[0,col].set_ylabel('Error ($)'); axes[0,col].grid(alpha=.3)
    axes[1,col].hist(res,bins=35,color=color,alpha=.75,edgecolor='white')
    axes[1,col].axvline(0,color='k',lw=1.5,ls='--')
    axes[1,col].axvline(res.mean(),color='red',lw=1.5,label=f'μ={res.mean():.2f}')
    axes[1,col].set_title('Error Distribution',fontsize=10,fontweight='bold')
    axes[1,col].set_xlabel('Error ($)'); axes[1,col].set_ylabel('Freq')
    axes[1,col].legend(fontsize=8); axes[1,col].grid(alpha=.3)
plt.tight_layout(); fig.savefig('/content/prediction5.png',dpi=110,bbox_inches='tight'); plt.close()

# ── P6: Metrics bar
fig,axes=plt.subplots(1,3,figsize=(13,4))
fig.suptitle('Model Comparison — Test Set Metrics',fontsize=13,fontweight='bold')
mns=rdf.index.tolist(); cols=[C['rnn'],C['lu'],C['lm']]
for ax,met in zip(axes,['RMSE','MAE','MAPE']):
    vals=[rdf.loc[m,met] for m in mns]
    bars=ax.bar(range(3),vals,color=cols,alpha=.85,edgecolor='white',width=.5)
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2,b.get_height()+max(vals)*.015,
                f'{v:.3f}',ha='center',va='bottom',fontsize=9,fontweight='bold')
    ax.set_title(met,fontsize=12,fontweight='bold')
    ax.set_xticks(range(3)); ax.set_xticklabels(['RNN','LSTM\nUni','LSTM\nMulti'],fontsize=9)
    ax.grid(alpha=.3,axis='y'); ax.set_facecolor('#f9f9f9'); ax.set_ylim(0,max(vals)*1.2)
plt.tight_layout(); fig.savefig('/content/prediction6.png',dpi=110,bbox_inches='tight'); plt.close()

# ── Master report ──────────────────────────
from matplotlib.image import imread
fig_m=plt.figure(figsize=(20,50)); fig_m.patch.set_facecolor('white')
paths=[f'/content/prediction{i}.png' for i in range(1,7)]
gs=gridspec.GridSpec(len(paths),1,figure=fig_m,hspace=.03)
for i,p in enumerate(paths):
    ax=fig_m.add_subplot(gs[i]); ax.imshow(imread(p)); ax.axis('off')
fig_m.text(.5,.998,'STOCK PRICE PREDICTION — RNN & LSTM FROM SCRATCH',
           ha='center',va='top',fontsize=21,fontweight='bold',color='#2c3e50')
fig_m.text(.5,.994,'Apple Inc. (AAPL)  |  NumPy Vectorised Implementation  |  Deep Learning Project',
           ha='center',va='top',fontsize=13,color='#7f8c8d')
fig_m.savefig('/content/prediction_report.png',dpi=100,bbox_inches='tight',facecolor='white')
plt.close()

# ── Copy outputs ─────────────────────────

shutil.copy('/content/prediction_report.png', '/content/lstm_stock.py')

print("\n" + "="*52)
print("  PROJECT COMPLETE!")
print("="*52)
print(rdf[['RMSE','MAE','MAPE','R2']].to_string())
print(f"\n  Best R²:   {rdf['R2'].idxmax()}")
print(f"  Best RMSE: {rdf['RMSE'].idxmin()}")

  STOCK PRICE PREDICTION — LSTM & RNN (NumPy from scratch)

[1/6] Generating AAPL-like stock data (Geometric Brownian Motion)…
   781 trading days  |  2020-01-28 → 2023-01-24
                  Open        High         Low       Close    Volume
2020-01-28  148.277674  148.840052  147.665159  148.226965  54102553
2020-01-29  146.118503  147.094826  143.602081  145.474074  78266442
2020-01-30  148.261903  148.744723  147.071849  148.581455  94674343
2020-01-31  149.107520  150.307269  146.556949  148.238912  63474284
2020-02-03  149.918501  149.918501  148.168697  148.499158  93293890

[2/6] Preprocessing & building sliding-window sequences…
   Seq len=30  |  Train=594  Test=314

[3/6] Training Simple RNN (batch-vectorized)…

[4/6] Training models…

   >>> Simple RNN (univariate)
   RNN  ep  10/40  train=0.00282  val=0.00125
   RNN  ep  20/40  train=0.00118  val=0.00114
   RNN  ep  30/40  train=0.00112  val=0.00108
   RNN  ep  40/40  train=0.00078  val=0.00075

   >>> LSTM Univariate
   L